[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeoLarryLai/pyPetrograph/blob/main/pyPetrograph_2_mineralmap_count.ipynb)


# pyPetrograph — mineral-map grain counter (v0.0.7)

Demo on three Rich Kyle–BEG test maps in `example_data/`: **10A**, **10C**, **10E**, plus `Mineral_Legend.tiff`.

These maps are already painted one solid color per mineral (white = background). The notebook does **not** use the RGB wand/LightGBM pipeline or the PPL/XPL object pipeline. No TensorFlow, no Qt.

Run cells **in order**. Outputs go to `outputs/mineralmap/`.

![Mineral-map workflow](image/workflow.png)

| Setting | What it does |
|---|---|
| `COLOR_TOL = 40` | Pixel belongs to a class if its RGB is within 40 of that legend color. |
| `SPECK_PX = 30` | Blobs smaller than 30 px are absorbed into the neighbor. |
| `UNKNOWN_SPECK_PX = 50` | Extra absorb for Unknown/black blobs up to 50 px. |
| `MIN_GRAIN_PX = 100` | Grains smaller than 100 px are dropped from the saved table. |
| `SPLIT = True` | Cut same-color grains that touch. A color change is always a boundary. |
| `BRIDGE_PX = 4` | Break bridges thinner than ~9 px. |
| `H_MIN = 3` | Cut a thin pinch between two bulky parts. The pinch must be ≥ 3 px closer to the edge than either bulge. |
| `NOTCH_SOLIDITY = 0.85`, `NOTCH_HALVES = 0.88` | V / L / needle cuts; both halves must be compact. `NOTCH_SOLIDITY = 0` turns this off. |
| `UNKNOWN_FRAC = 0.95` | Grain is Unknown only if ≥ 95% of its pixels are Unknown/black. |
| `DISPLAY_RGB` | Overlay-only color overrides (Biotite mid-blue here). |


## Setup

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

ROOT = Path(".")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

from pyPetrograph import check_and_install_packages
_ = check_and_install_packages(install=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 150
from PIL import Image

from pyPetrograph import (
    set_output_dir,
    read_legend_swatches,
    legend_from_image,
    check_legend,
    process_folder,
    format_counts,
    format_unlisted,
    pivot_counts,
    mineralmap_paths,
)


## Settings

In [ ]:
set_output_dir("outputs")

IMAGE_DIR = Path("example_data")
LEGEND_PATH = IMAGE_DIR / "Mineral_Legend.tiff"
DEMO_STEMS = ["10A_Mineral_Image", "10C_Mineral_Image", "10E_Mineral_Image"]
IMAGE_PATHS = [IMAGE_DIR / f"{s}.tiff" for s in DEMO_STEMS]

LEGEND = {
    "Monazite & Xenotime": (255, 0, 0),
    "Zircon": (0, 255, 255),
    "Rutile": (255, 192, 203),
    "Ilmenite": (0, 100, 0),
    "Iron Oxides": (128, 0, 0),
    "Biotite": (0, 0, 85),
    "Quartz": (255, 255, 0),
    "Chromite": (255, 0, 255),
    "Orthoclase": (255, 165, 0),
    "Carpholite": (205, 92, 92),
    "Garnet-Spessartine": (106, 90, 205),
    "Pseudobrookite": (0, 255, 0),
    "Illite": (184, 134, 11),
    "Calderite": (227, 92, 28),
    "Topaz": (148, 0, 211),
    "Tourmaline": (65, 105, 225),
    "Unknown": (0, 0, 0),
}

# Or build LEGEND from the swatch image (names must match swatches top to bottom):
# LEGEND = legend_from_image(LEGEND_PATH, names=[
#     "Monazite & Xenotime", "Zircon", "Rutile", "Ilmenite", "Iron Oxides",
#     "Biotite", "Quartz", "Chromite", "Orthoclase", "Carpholite",
#     "Garnet-Spessartine", "Pseudobrookite", "Illite", "Calderite",
#     "Topaz", "Tourmaline",
# ])

COLOR_TOL = 40
SPECK_PX = 30
UNKNOWN_SPECK_PX = 50
MIN_GRAIN_PX = 100
SPLIT = True
BRIDGE_PX = 4
H_MIN = 3.0
NOTCH_SOLIDITY = 0.85
NOTCH_HALVES = 0.88
UNKNOWN_FRAC = 0.95
DISPLAY_RGB = {"Biotite": (70, 130, 255)}  # overlay only

N_JOBS = 4
LOAD_SAVED = True
WRITE_GEOJSON = True

print(f"{len(IMAGE_PATHS)} images:", [p.stem for p in IMAGE_PATHS])
for p in IMAGE_PATHS:
    if not p.exists():
        raise FileNotFoundError(p)


## 1. Legend check

In [ ]:
swatches = read_legend_swatches(LEGEND_PATH)
problems = check_legend(LEGEND, swatches)
if problems:
    print("LEGEND mismatches:")
    for m in problems:
        print("  -", m)
else:
    print(f"LEGEND matches {len(swatches)} swatches in {LEGEND_PATH.name}")


## 2. Run (w/ uncertainty estimate)

Each cell is **grain count ± counting noise [low–high]**.

- The middle number is the count with the settings above.
- ± is typical Poisson scatter (√n).
- [low–high] is the count if you make the smallest kept grain twice as small or twice as large.


In [ ]:
all_counts = process_folder(
    IMAGE_PATHS,
    LEGEND,
    n_jobs=N_JOBS,
    load_saved=LOAD_SAVED,
    color_tol=COLOR_TOL,
    speck_px=SPECK_PX,
    unknown_speck_px=UNKNOWN_SPECK_PX,
    min_grain_px=MIN_GRAIN_PX,
    split=SPLIT,
    bridge_px=BRIDGE_PX,
    h_min=H_MIN,
    notch_solidity=NOTCH_SOLIDITY,
    notch_halves=NOTCH_HALVES,
    write_geojson=WRITE_GEOJSON,
    unknown_frac=UNKNOWN_FRAC,
    display_rgb=DISPLAY_RGB,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
display(format_counts(all_counts).reindex(list(LEGEND) + ["TOTAL"]))


## 3. Preview (10A, 10C, 10E)

Left: the original painted map. Right: the same grains colored by the legend class we assigned.

The table under each pair lists map colors that did **not** match any legend color (farther than `COLOR_TOL`). Those pixels were counted as **Unknown**.

| Column | Meaning |
|---|---|
| `map_red`, `map_green`, `map_blue` | That color on the map |
| `pixels` | How many pixels have that color |
| `closest_legend_color` | Nearest name in the legend (still too far to use) |
| `distance_from_legend` | How far that color is from that legend swatch, in RGB units. 0 = exact match. Every row here is greater than `COLOR_TOL` (40). |


In [ ]:
for PREVIEW in IMAGE_PATHS:
    paths = mineralmap_paths(PREVIEW)
    if not paths["class_rgb"].exists():
        print("skip preview — missing", paths["class_rgb"])
        continue
    orig = Image.open(PREVIEW).convert("RGB")
    over = Image.open(paths["class_rgb"])
    scale = 800 / max(orig.size)
    size = (int(orig.size[0] * scale), int(orig.size[1] * scale))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=150)
    axes[0].imshow(orig.resize(size, Image.NEAREST)); axes[0].set_title(PREVIEW.stem)
    axes[1].imshow(over.resize(size, Image.NEAREST)); axes[1].set_title("grains by class")
    for ax in axes:
        ax.set_axis_off()
    plt.tight_layout(); plt.show()

    unlisted = format_unlisted(pd.read_csv(paths["unlisted_csv"]))
    print(
        f"{len(unlisted)} map colors in {PREVIEW.stem} were farther than COLOR_TOL={COLOR_TOL} "
        "from every legend swatch, so they were counted as Unknown."
    )
    print("distance_from_legend = how far that map color is from the closest legend color (RGB units; 0 = exact match).")
    display(unlisted.head(15))


## 4. Share of grains

Stacked bars: percent of grains in each mineral class (including Unknown). The table under the chart is the raw grain counts.


In [ ]:
share = pivot_counts(all_counts[all_counts["class"] != "TOTAL"], "share_pct").fillna(0.0)
share = share.reindex(list(LEGEND)).fillna(0.0)
colors = [tuple(v / 255 for v in LEGEND.get(c, (128, 128, 128))) for c in share.index]

fig, ax = plt.subplots(figsize=(8, 3.5), dpi=150)
bottom = np.zeros(share.shape[1])
for (cls, row), col in zip(share.iterrows(), colors):
    ax.bar(share.columns, row.to_numpy(), bottom=bottom, color=col, edgecolor="k", linewidth=0.3, label=cls)
    bottom += row.to_numpy()
ax.set_ylabel("share of grains (%)")
ax.set_xticks(range(share.shape[1]))
ax.set_xticklabels([c.replace("_Mineral_Image", "") for c in share.columns], rotation=45, ha="right")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

counts_n = pivot_counts(all_counts, "n").reindex(list(LEGEND) + ["TOTAL"])
counts_n.columns = [c.replace("_Mineral_Image", "") for c in counts_n.columns]
display(counts_n.astype("Int64"))
